In [38]:
# volatility, taker buy/sell ratio, momentum, auto correlation, level

import polars as pl

vol_window = 30
yz_k = 0.34 / (1.34 + (vol_window + 1) / (vol_window - 1))

df = pl.read_parquet('binance-1d-spot-stable-pairs.parquet').sort(['symbol', 'ts']).with_columns([
    # yang-zhang volatility
    (
        # overnight term
        (pl.col('open') / pl.col('close').shift(1)).log()#+.rolling_var(window_size=30).over('symbol') +
        # open-close term
        (yz_k * (pl.col('close') / pl.col('open')).log()+#.rolling_var(window_size=30).over('symbol')) +
        # rogers-satchell term
        ((1 - yz_k) * (
            ((pl.col('high') / pl.col('close')).log() * (pl.col('high') / pl.col('open')).log()) +
            ((pl.col('low') / pl.col('close')).log() * (pl.col('low') / pl.col('open')).log())
        )#.rolling_mean(30).over('symbol'))
    ).sqrt().alias('sigma_yz'),

    # todays log return
    (pl.col('close') / pl.col('close').shift(1)).log().over('symbol').alias('ret'),
])

ts,pair,open,high,low,close,volume,taker_buy_quote_asset_volume,taker_buy_base_asset_volume,symbol,sigma_yz,ret
date,str,f64,f64,f64,f64,f64,f64,f64,str,f64,f64
2025-09-22,"""0gusdt""",1.0,7.26,1.0,4.839,6.9412e7,1.7425e8,3.4993e7,"""0g""",null,null
2025-09-23,"""0gusdt""",4.838,7.18,4.566,5.815,5.2750e7,1.5416e8,2.6663e7,"""0g""",0.330558,0.183733
2025-09-24,"""0gusdt""",5.82,5.895,4.838,5.005,1.9527e7,4.5399e7,8.6474e6,"""0g""",NaN,-0.150003
2025-09-25,"""0gusdt""",5.006,5.014,3.649,3.917,1.3282e7,2.6296e7,6.5139e6,"""0g""",NaN,-0.245111
2025-09-26,"""0gusdt""",3.917,4.455,3.341,3.679,1.4793e7,2.8256e7,7.3628e6,"""0g""",0.159602,-0.062685
…,…,…,…,…,…,…,…,…,…,…,…
2025-11-26,"""zrxusdt""",0.1592,0.16,0.1523,0.1595,1.2282336e7,957961.7881,6.115821e6,"""zrx""",0.045133,0.001883
2025-11-27,"""zrxusdt""",0.1596,0.1681,0.1578,0.1645,1.5357909e7,1.2407e6,7.6249e6,"""zrx""",0.079149,0.030867
2025-11-28,"""zrxusdt""",0.1645,0.1674,0.1593,0.16,1.4493869e7,1.1969e6,7.246667e6,"""zrx""",NaN,-0.027737


In [35]:
from arch import arch_model

rets = df.filter([pl.col('symbol') == 'btc'])[['ts','ret', 'sigma_yz']].drop_nulls().tail(1000)
gjr = arch_model(rets['ret'].to_numpy() * 100, vol='Garch', p=1, o=1, q=1, dist='skewt', rescale=False).fit()

rets = rets.with_columns([
    pl.Series('garch', gjr.conditional_volatility / 100)
])
rets.select([
    ((pl.col("garch") - pl.col("sigma_yz"))**2).mean().sqrt().alias("rmse"),
    (pl.col("garch") - pl.col("sigma_yz")).abs().mean().alias("mae"),    
    # Bias: Mean(Error) - tells you if GARCH consistently over/under predicts
    (pl.col("garch") - pl.col("sigma_yz")).mean().alias("bias")
])
rets

Iteration:      1,   Func. Count:      9,   Neg. LLF: 30919.17324278717
Iteration:      2,   Func. Count:     19,   Neg. LLF: 109874.6680428356
Iteration:      3,   Func. Count:     29,   Neg. LLF: 40830.95997503287
Iteration:      4,   Func. Count:     39,   Neg. LLF: 3438.5853503699846
Iteration:      5,   Func. Count:     48,   Neg. LLF: 2920.1548157417874
Iteration:      6,   Func. Count:     57,   Neg. LLF: 2942.3030514506645
Iteration:      7,   Func. Count:     66,   Neg. LLF: 2937.7682970427923
Iteration:      8,   Func. Count:     75,   Neg. LLF: 2835.664097092133
Iteration:      9,   Func. Count:     84,   Neg. LLF: 2884.988019878039
Iteration:     10,   Func. Count:     93,   Neg. LLF: 2887.2927878065584
Iteration:     11,   Func. Count:    102,   Neg. LLF: 2726.664499996097
Iteration:     12,   Func. Count:    111,   Neg. LLF: 2885.9621369076012
Iteration:     13,   Func. Count:    120,   Neg. LLF: 2672.713664162843
Iteration:     14,   Func. Count:    129,   Neg. LLF: 2221

ts,ret,sigma_yz,garch
date,f64,f64,f64
2023-03-07,-0.009507,NaN,0.037773
2023-03-08,-0.022437,NaN,0.036694
2023-03-09,-0.063882,NaN,0.036203
2023-03-10,-0.010443,NaN,0.039977
2023-03-11,0.015025,0.054254,0.038819
…,…,…,…
2025-11-26,0.035022,0.074164,0.028639
2025-11-27,0.009349,0.038258,0.028618
2025-11-28,-0.004865,NaN,0.027867
